# Prerequisites

In [55]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [56]:
# Auto-reload module eksternal setiap cell dijalankan ulang
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Load file artikel

In [57]:
from pathlib import Path
import pandas as pd

article_path = Path("article.md")
article = article_path.read_text(encoding="utf-8")

print(f"Loaded: {article_path} ({len(article)} karakter)")
print(article)  # preview

Loaded: article.md (7250 karakter)
# What is Anthropic's Claude Mythos and what risks does it pose?

16 hours ago

Liv McMahon,Technology reporterand

Joe Tidy,Cyber correspondent, BBC World Service

![](https://static.files.bbci.co.uk/bbcdotcom/web/20260407-092955-f3cfe0ee04-web-3.0.0-2/grey-placeholder.png)!Reuters A smartphone display showing the Anthropic logo in black letters on an all-white background, laid on a laptop keyboard lit in pink and purpleReuters

In recent weeks, the AI world has been a-buzz following claims made by leading firm, Anthropic, regarding its new model, Claude Mythos.

The company says it found the tool can outperform humans at some hacking and cyber-security tasks, which has prompted discussions by regulators, legislators and financial institutions about the dangers it could pose to digital services.

Several tech giants have been given access to Mythos via an initiative called Project Glasswing, designed to strengthen resilience to Mythos itself.

But ot

# Proses

## Pecah menjadi beberapa paragraph

In [58]:
import importlib
import preprocess

# Pastikan module eksternal memakai versi terbaru dari file .py
importlib.reload(preprocess)

# ambil query dari heading pertama markdown
first_nonempty_line = next((line.strip() for line in article.splitlines() if line.strip()), "")
# print(f"First non-empty line: {first_nonempty_line}")
query = first_nonempty_line.lstrip("#").strip()
print(f"Query: {query}")
query_tokens = preprocess.preProcess(query)
print(f"Query Tokens: {query_tokens}\n")

paragraphs = preprocess.splitParagraph(article)
print(f"Split into {len(paragraphs)} paragraphs")


Query: What is Anthropic's Claude Mythos and what risks does it pose?
Query Tokens: ['anthrop', 'claud', 'mytho', 'risk', 'pose']

Split into 5 paragraphs


[nltk_data] Downloading package punkt to /home/juana/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Proses setiap Paragraph

In [ ]:
import preprocess
import counter

# ===============================
# STORAGE
# ===============================
tf_global = {}          # TF semua dokumen
tf_paragraph = {}       # TF per paragraph
df_all = {}             # DF per paragraph
idf_paragraph = {}      # IDF per paragraph
tfidf_paragraph = {}    # TF-IDF per paragraph


doc_counter = 1

# ===============================
# MAIN LOOP
# ===============================
for i, (title, content) in enumerate(paragraphs.items(), start=1):
    
    paragraph_key = f"P{i}"
    print(f"Processing {paragraph_key}/{len(paragraphs)} : {title}")
    
    docs = preprocess.splitDocument(content)
    print(f"  Split into {len(docs)} documents")
    # -------------------------------
    # PREPROCESS ALL DOCS
    # -------------------------------
    clean_docs = []
    paragraph_tf = {}


    
    # hitung tf dan idf query
    doc_key = "QUERY"
    tf_entry = {
        term: counter.tf_counter(term, query_tokens)
        for term in query_tokens
    }
    tf_global[doc_key] = tf_entry
    paragraph_tf[doc_key] = tf_entry
        
        
    for doc in docs:
        clean_doc = preprocess.preProcess(doc)
        clean_docs.append(clean_doc)

        doc_key = f"D{doc_counter}"

        # Hitung TF
        tf_entry = {
            term: counter.tf_counter(term, clean_doc)
            for term in query_tokens
        }

        tf_global[doc_key] = tf_entry
        paragraph_tf[doc_key] = tf_entry

        doc_counter += 1

    # -------------------------------
    # HITUNG DF & IDF
    # -------------------------------
    N = len(clean_docs)

    df_terms = {}
    idf_terms = {}

    for term in query_tokens:
        df_value = counter.calc_df(term, clean_docs)
        df_terms[term] = df_value

        idf_terms[term] = (
            counter.calc_idf(N, df_value)
            if df_value > 0 else 0
        )

    # -------------------------------
    # SIMPAN HASIL
    # -------------------------------
    tf_paragraph[paragraph_key] = paragraph_tf
    df_all[paragraph_key] = df_terms
    idf_paragraph[paragraph_key] = idf_terms

    # -------------------------------
    # HITUNG TF-IDF
    # -------------------------------
    tfidf_paragraph[paragraph_key] = {
        doc_key: {
            term: paragraph_tf[doc_key][term] * idf_terms[term]
            for term in query_tokens
        }
        for doc_key in paragraph_tf
    }
    


Processing P1/5 : 1
  Split into 4 documents
Processing P2/5 : 2
  Split into 8 documents
Processing P3/5 : 3
  Split into 10 documents
Processing P4/5 : 4
  Split into 6 documents
Processing P5/5 : 5
  Split into 10 documents


In [60]:
# print(tf_paragraph)
# print(tfidf_paragraph)
# print(df_all)
for para_key, docs in tfidf_paragraph.items():
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    print("\n[TF]")
    display(pd.DataFrame(tf_paragraph[para_key]).round(3))
    
    print("\n[DF]")
    display(pd.Series(df_all[para_key]).round(3).to_frame("df"))
    
    print("\n[IDF]")
    display(pd.Series(idf_paragraph[para_key]).round(3).to_frame("idf"))
    
    print("\n[TF-IDF]")
    display(pd.DataFrame(tfidf_paragraph[para_key]).round(3))
    
    # print("\n[VSM Cosine Similarity]")
    # para_vsm = {dk: vsm_scores[dk] for dk in docs}
    # display(pd.Series(para_vsm).round(3).to_frame("score"))


  P1

[TF]


,QUERY,D1,D2,D3,D4
anthrop,1,1,0,0,1
claud,1,1,0,0,0
mytho,1,1,0,2,0
risk,1,0,0,0,0
pose,1,0,1,0,0



[DF]


,df
anthrop,2
claud,1
mytho,2
risk,0
pose,1



[IDF]


...

,df
anthrop,2
claud,0
mytho,1
risk,1
pose,0



[IDF]


,idf
anthrop,2.609
claud,0.000
mytho,3.303
risk,3.303
pose,0.000



[TF-IDF]


,QUERY,D29,D30,D31,D32,D33,D34,D35,D36,D37,D38
anthrop,2.609,0.0,0.0,0.0,0.000,2.609,0.0,0.0,0.0,2.609,0.0
claud,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0
mytho,3.303,0.0,0.0,0.0,3.303,0.000,0.0,0.0,0.0,0.000,0.0
risk,3.303,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,3.303,0.0
pose,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0


## hitung jarak dokumen dan query

akar sigma W ** 2 

In [61]:
jarak = {}

for para_key, docs in tfidf_paragraph.items():
    jarak_per_paragraph = {}
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    # print("\n[TF-IDF]")
    # display(pd.DataFrame(tfidf_paragraph[para_key]).round(3))
    print("akar sum kuadrat")
    for doc_key, W in tfidf_paragraph[para_key].items():
        sum_of_squares = sum(value ** 2 for value in W.values())
        magnitude = sum_of_squares ** 0.5
        
        print(f"{doc_key}: {magnitude:.3f}")
        jarak_per_paragraph[doc_key] = magnitude
    
    jarak[para_key] = jarak_per_paragraph


  P1
akar sum kuadrat
QUERY: 4.138
D1: 3.381
D2: 2.386
D3: 3.386
D4: 1.693

  P2
akar sum kuadrat
QUERY: 4.495
D5: 3.274
D6: 2.242
D7: 1.693
D8: 0.000
D9: 2.803
D10: 0.000
D11: 2.242
D12: 3.412

  P3
akar sum kuadrat
QUERY: 4.541
D13: 2.204
D14: 3.117
D15: 0.000
D16: 0.000
D17: 0.000
D18: 2.204
D19: 0.000
D20: 3.303
D21: 3.117
D22: 0.000

  P4
akar sum kuadrat
QUERY: 1.693
D23: 1.693
D24: 0.000
D25: 1.693
D26: 0.000
D27: 1.693
D28: 0.000

  P5
akar sum kuadrat
QUERY: 5.350
D29: 0.000
D30: 0.000
D31: 0.000
D32: 3.303
D33: 2.609
D34: 0.000
D35: 0.000
D36: 0.000
D37: 4.209
D38: 0.000


In [62]:
for para_key, docs in jarak.items():
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    print("\n[Jarak]")
    display(pd.Series(docs).round(3).to_frame("jarak").T)


  P1

[Jarak]


,QUERY,D1,D2,D3,D4
jarak,4.138,3.381,2.386,3.386,1.693



  P2

[Jarak]


,QUERY,D5,D6,D7,D8,D9,D10,D11,D12
jarak,4.495,3.274,2.242,1.693,0.0,2.803,0.0,2.242,3.412



  P3

[Jarak]


,QUERY,D13,D14,D15,D16,D17,D18,D19,D20,D21,D22
jarak,4.541,2.204,3.117,0.0,0.0,0.0,2.204,0.0,3.303,3.117,0.0



  P4

[Jarak]


,QUERY,D23,D24,D25,D26,D27,D28
jarak,1.693,1.693,0.0,1.693,0.0,1.693,0.0



  P5

[Jarak]


,QUERY,D29,D30,D31,D32,D33,D34,D35,D36,D37,D38
jarak,5.35,0.0,0.0,0.0,3.303,2.609,0.0,0.0,0.0,4.209,0.0


In [63]:
print(tfidf_paragraph)
for para_key, docs in tfidf_paragraph.items():
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    display(pd.DataFrame(tfidf_paragraph[para_key]).round(3))

{'P1': {'QUERY': {'anthrop': 1.6931471805599454, 'claud': 2.386294361119891, 'mytho': 1.6931471805599454, 'risk': 0, 'pose': 2.386294361119891}, 'D1': {'anthrop': 1.6931471805599454, 'claud': 2.386294361119891, 'mytho': 1.6931471805599454, 'risk': 0, 'pose': 0.0}, 'D2': {'anthrop': 0.0, 'claud': 0.0, 'mytho': 0.0, 'risk': 0, 'pose': 2.386294361119891}, 'D3': {'anthrop': 0.0, 'claud': 0.0, 'mytho': 3.386294361119891, 'risk': 0, 'pose': 0.0}, 'D4': {'anthrop': 1.6931471805599454, 'claud': 0.0, 'mytho': 0.0, 'risk': 0, 'pose': 0.0}}, 'P2': {'QUERY': {'anthrop': 1.4700036292457357, 'claud': 2.386294361119891, 'mytho': 1.6931471805599454, 'risk': 3.0794415416798357, 'pose': 0}, 'D5': {'anthrop': 1.4700036292457357, 'claud': 2.386294361119891, 'mytho': 1.6931471805599454, 'risk': 0.0, 'pose': 0}, 'D6': {'anthrop': 1.4700036292457357, 'claud': 0.0, 'mytho': 1.6931471805599454, 'risk': 0.0, 'pose': 0}, 'D7': {'anthrop': 0.0, 'claud': 0.0, 'mytho': 1.6931471805599454, 'risk': 0.0, 'pose': 0}, '

,QUERY,D1,D2,D3,D4
anthrop,1.693,1.693,0.000,0.000,1.693
claud,2.386,2.386,0.000,0.000,0.000
mytho,1.693,1.693,0.000,3.386,0.000
risk,0.000,0.000,0.000,0.000,0.000
pose,2.386,0.000,2.386,0.000,0.000



  P2


,QUERY,D5,D6,D7,D8,D9,D10,D11,D12
anthrop,1.470,1.470,1.470,0.000,0.0,1.470,0.0,1.470,1.470
claud,2.386,2.386,0.000,0.000,0.0,2.386,0.0,0.000,0.000
mytho,1.693,1.693,1.693,1.693,0.0,0.000,0.0,1.693,0.000
risk,3.079,0.000,0.000,0.000,0.0,0.000,0.0,0.000,3.079
pose,0.000,0.000,0.000,0.000,0.0,0.000,0.0,0.000,0.000



  P3


,QUERY,D13,D14,D15,D16,D17,D18,D19,D20,D21,D22
anthrop,2.204,2.204,2.204,0.0,0.0,0.0,0.000,0.0,0.000,2.204,0.0
claud,0.000,0.000,0.000,0.0,0.0,0.0,0.000,0.0,0.000,0.000,0.0
mytho,2.204,0.000,2.204,0.0,0.0,0.0,2.204,0.0,0.000,2.204,0.0
risk,3.303,0.000,0.000,0.0,0.0,0.0,0.000,0.0,3.303,0.000,0.0
pose,0.000,0.000,0.000,0.0,0.0,0.0,0.000,0.0,0.000,0.000,0.0



  P4


,QUERY,D23,D24,D25,D26,D27,D28
anthrop,0.000,0.000,0.0,0.000,0.0,0.000,0.0
claud,0.000,0.000,0.0,0.000,0.0,0.000,0.0
mytho,1.693,1.693,0.0,1.693,0.0,1.693,0.0
risk,0.000,0.000,0.0,0.000,0.0,0.000,0.0
pose,0.000,0.000,0.0,0.000,0.0,0.000,0.0



  P5


,QUERY,D29,D30,D31,D32,D33,D34,D35,D36,D37,D38
anthrop,2.609,0.0,0.0,0.0,0.000,2.609,0.0,0.0,0.0,2.609,0.0
claud,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0
mytho,3.303,0.0,0.0,0.0,3.303,0.000,0.0,0.0,0.0,0.000,0.0
risk,3.303,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,3.303,0.0
pose,0.000,0.0,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.0


## hitung dot product


In [64]:
dot_product = {}
for para_key, docs in tfidf_paragraph.items():
    # print(f"\n{'='*40}")
    # print(f"  {para_key}")
    # print(f"{'='*40}")
    # display(pd.DataFrame(tfidf_paragraph[para_key]).round(3))
    dot_product_paragraph = {}
    for doc_key, W in tfidf_paragraph[para_key].items():
        # print(f"Document: {doc_key}")
        # print(f"TF-IDF Vector: {W}")
        Q = tfidf_paragraph[para_key].get("QUERY", {})
        # print(f"Query TF-IDF Vector: {Q}")
        dot_product_value = sum(W.get(term, 0) * Q.get(term, 0) for term in Q)
        dot_product_paragraph[doc_key] = dot_product_value
        
    dot_product[para_key] = dot_product_paragraph
    
print(dot_product)

{'P1': {'QUERY': 17.122296305901358, 'D1': 11.427895527988772, 'D2': 5.694400777912588, 'D3': 5.733494750076185, 'D4': 2.8667473750380923}, 'P2': {'QUERY': 20.205019031569798, 'D5': 10.722058822946314, 'D6': 5.027658045033727, 'D7': 2.8667473750380923, 'D8': 0.0, 'D9': 7.855311447908222, 'D10': 0.0, 'D11': 5.027658045033727, 'D12': 11.643870878619119}, 'P3': {'QUERY': 20.62206054088315, 'D13': 4.85749612220833, 'D14': 9.71499224441666, 'D15': 0.0, 'D16': 0.0, 'D17': 0.0, 'D18': 4.85749612220833, 'D19': 0.0, 'D20': 10.907068296466491, 'D21': 9.71499224441666, 'D22': 0.0}, 'P4': {'QUERY': 2.8667473750380923, 'D23': 2.8667473750380923, 'D24': 0.0, 'D25': 2.8667473750380923, 'D26': 0.0, 'D27': 2.8667473750380923, 'D28': 0.0}, 'P5': {'QUERY': 28.623302811781418, 'D29': 0.0, 'D30': 0.0, 'D31': 0.0, 'D32': 10.907068296466491, 'D33': 6.8091662188484365, 'D34': 0.0, 'D35': 0.0, 'D36': 0.0, 'D37': 17.716234515314927, 'D38': 0.0}}


In [ ]:
cosine_similarity = {}
print("\n[dot product]")
for para_key, docs in dot_product.items():
    cosine_similarity_per_paragraph = {}
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    # print(docs)
    for term in docs:
        
        similarity = counter.calc_cosine_similarity(docs[term], jarak[para_key].get(term, 0.0), jarak[para_key].get("QUERY", 0.0))
        # print(f"{similarity:.3f}")
        cosine_similarity_per_paragraph[term] = similarity

    cosine_similarity[para_key] = cosine_similarity_per_paragraph

    # display(pd.Series(cosine_similarity_per_paragraph).round(3).to_frame("cosine_similarity").T)


[dot product]

  P1


,QUERY,D1,D2,D3,D4
cosine_similarity,1.0,0.817,0.577,0.409,0.409



  P2


,QUERY,D5,D6,D7,D8,D9,D10,D11,D12
cosine_similarity,1.0,0.728,0.499,0.377,0.0,0.624,0.0,0.499,0.759



  P3


,QUERY,D13,D14,D15,D16,D17,D18,D19,D20,D21,D22
cosine_similarity,1.0,0.485,0.686,0.0,0.0,0.0,0.485,0.0,0.727,0.686,0.0



  P4


,QUERY,D23,D24,D25,D26,D27,D28
cosine_similarity,1.0,1.0,0.0,1.0,0.0,1.0,0.0



  P5


,QUERY,D29,D30,D31,D32,D33,D34,D35,D36,D37,D38
cosine_similarity,1.0,0.0,0.0,0.0,0.617,0.488,0.0,0.0,0.0,0.787,0.0


# ranking dokumen

In [76]:
top_2_results = {}  # Wadah untuk menyimpan hasil akhir

for para_key, docs in jarak.items():
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    # Hapus QUERY agar tidak ikut terhitung
    cosine_similarity[para_key].pop("QUERY", None)
    
    # Urutkan berdasarkan skor tertinggi
    rank = dict(sorted(cosine_similarity[para_key].items(), key=lambda item: item[1], reverse=True))
    
    # Ambil 2 teratas menggunakan slicing list dan ubah kembali ke dict
    top_2 = dict(list(rank.items())[:2])
    top_2_results[para_key] = top_2 # Simpan ke variabel luar
    
    print("\n[Top 2 Rank]")
    for term, score in top_2.items():
        print(f"  {term}: {score:.3f}")

# Sekarang variabel 'top_2_results' berisi dokumen terbaik untuk setiap para_key



  P1

[Top 2 Rank]
  D1: 0.817
  D2: 0.577

  P2

[Top 2 Rank]
  D12: 0.759
  D5: 0.728

  P3

[Top 2 Rank]
  D20: 0.727
  D14: 0.686

  P4

[Top 2 Rank]
  D23: 1.000
  D25: 1.000

  P5

[Top 2 Rank]
  D37: 0.787
  D32: 0.617
